In [1]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt

%matplotlib inline
%matplotlib notebook

### 1.Load file.

#### 1.1 Load csv file using 'pandas.read_csv'

In [2]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [3]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

In [4]:
#print(InputData)

#### 1.2 Load csv file using ''csv'.

In [5]:
import csv
with open('input_csv.csv') as f:
    M = list(csv.reader(f, delimiter=','))
    #reader = csv.reader(f, delimiter=',')
    #for row in reader:
        #print(row)
print(M[20])

['1/1/1991 19:00', '0', '0', '0']


### 2. Paved roof ###

#### 2.1 Try ''while' iterator (with arrays)
#### Advantage: Straightforward to program;
#### Disadvantage: Array ---> memory-consuming

In [6]:
iters = np.shape(P_atm)[0] # total timestep.

In [7]:
PR_measure = 0 # we do not consider 'measure' for the time being.

If using np.savetxt() to save csv file, and save all results after simulation.

In [8]:
# This one can write result vertically in csv file 
IntStor_PR = np.zeros((iters,1))
E_atm_PR = np.zeros((iters,1))
Int_PR = np.zeros((iters,1))
R_swds_PR = np.zeros((iters,1))
R_mss_PR = np.zeros((iters,1))
R_measure_PR = np.zeros((iters,1)) # Not consider for the time being, all zeros.

IntStorCap_PR = 1.6  # InstorCap is defined as 1.6mm.  ----- parameter
Disc_partial_PR = 0  # Disconnected part is defined as 0%. ---- parameter
Storm_partial_PR = 1.0 # Stormwater part is defined as 100% ---- parameter
Mix_partial_PR = 1 - Storm_partial_PR

t = 1
while t <= iters-1:  
    # for PR I,E, Istor component only.
    Int_PR[t] = np.minimum(IntStorCap_PR, np.maximum(0, P_atm[t] + IntStor_PR[t-1]))
    E_atm_PR[t] = np.minimum(E_pot_OW[t], Int_PR[t])
    IntStor_PR[t] = Int_PR[t] - E_atm_PR[t] # This part is the same as the excel, different from the description file.
    R_swds_PR[t] = Storm_partial_PR * (1 - Disc_partial_PR) * np.maximum(0, P_atm[t] - E_atm_PR[t] -(IntStor_PR[t] - IntStor_PR[t-1]) - R_measure_PR[t])
    R_mss_PR[t] = Mix_partial_PR * (1 - Disc_partial_PR) * np.maximum(0, P_atm[t] - E_atm_PR[t] -(IntStor_PR[t] - IntStor_PR[t-1]) - R_measure_PR[t])
    t += 1
#print(E_atm_PR)
#print(R_swds_PR)
np.savetxt('sol/IntStor_sol.csv', (IntStor_PR), fmt='%.18e', delimiter=',')
np.savetxt('sol/E_atm_PR_sol.csv', (E_atm_PR), fmt='%.18e', delimiter=',')
np.savetxt('sol/Int_PR_sol.csv', (Int_PR), fmt='%.18e', delimiter=',')
np.savetxt('sol/R_swds_PR.csv', (R_swds_PR), fmt='%.18e', delimiter=',')
np.savetxt('sol/R_mss_PR.csv', (R_swds_PR), fmt='%.18e', delimiter=',')

In [9]:
# This one can write three results horizontally in csv file.
IntStor_PR = np.zeros(iters)
E_atm_PR = np.zeros(iters)
Int_PR = np.zeros(iters)
R_swds_PR = np.zeros(iters)
R_mss_PR = np.zeros(iters)
R_measure_PR = np.zeros(iters) # Not consider for the time being, all zeros.

IntStorCap_PR = 1.6  # InstorCap is defined as 1.6mm.  ----- parameter
Disc_partial_PR = 0  # Disconnected part is defined as 0%. ---- parameter
Storm_partial_PR = 1.0 # Stormwater part is defined as 100% ---- parameter
Mix_partial_PR = 1 - Storm_partial_PR

t = 1
while t <= iters-1:  
    Int_PR[t] = np.minimum(IntStorCap_PR, np.maximum(0, P_atm[t] + IntStor_PR[t-1]))
    E_atm_PR[t] = np.minimum(E_pot_OW[t], Int_PR[t])
    IntStor_PR[t] = Int_PR[t] - E_atm_PR[t] # This part is the same as the excel, different from the description file.
    R_swds_PR[t] = Storm_partial_PR * (1 - Disc_partial_PR) * np.maximum(0, P_atm[t] - E_atm_PR[t] -(IntStor_PR[t] - IntStor_PR[t-1]) - R_measure_PR[t])
    R_mss_PR[t] = Mix_partial_PR * (1 - Disc_partial_PR) * np.maximum(0, P_atm[t] - E_atm_PR[t] -(IntStor_PR[t] - IntStor_PR[t-1]) - R_measure_PR[t])
    t += 1
#print(np.reshape(E_atm_PR,(iters,1)))
np.savetxt('sol/results.csv', (Int_PR, E_atm_PR, IntStor_PR, R_swds_PR, R_mss_PR), fmt='%.18e', delimiter=',')

If using q = [], and append().

If using csv_writerow to save csv file. Can we write down results after each time step? 

#### 2.2 Try Class, and '\__init\__' iterator and '\_next()\_'

Class Function below solves for the solutions at time level t, based on the input(P and E) at time level t and output at time level t -1.

In [93]:
min(3.0,4.5)

3.0

### Modify later

In [96]:
# np.minimum() to min()
# lowercase for variables and CamelCase for class name

class PavedRoof:
    def __init__(self, t, P_atm , E_pot_OW, Init_InStor, IntStorCap_PavedRoof = 1.6, StormFrac_PavedRoof = 1.0, DiscFrac_PavedRoof = 0.0):
        # self.P_atm = P_atm
        
        # state
        self.init_InStor = Init_InStor
        
        # parameters
        self.IntStorCap = IntStorCap_PavedRoof
        self.StormFrac = StormFrac_PavedRoof
        # self.MxdFrac = 1 - StormFrac_PavedRoof
        self.DiscFrac = DiscFrac_PavedRoof
        
    def __repr__(self):
        return 'Current P is ' + str(self.P_atm) + 'Current E is ' + str(self.E_pot_OW) + '.These are input information.'
        
    def mxd_frac(self):
        return 1 - StormFrac_PavedRoof
        
    def sol(self, ):
        Int = np.minimum(self.IntStorCap, np.maximum(0, self.P_atm + self.init_InStor))
        E_atm = np.minimum( self.E_pot_OW, Int)
        IntStor = Int - E_atm
        R_swds = self.StormFrac * (1 - self.DiscFrac) * np.maximum(0, self.P_atm - E_atm -( IntStor - self.init_InStor) )
        R_mss = self.MxdFrac * (1 - self.DiscFrac) * np.maximum(0, self.P_atm - E_atm - ( IntStor - self.init_InStor))
        R_up = self.DiscFrac * np.maximum(0, self.P_atm - E_atm - (IntStor - self.init_InStor))
        
        # update state
        self.init_InStor = IntStor
        
        return Int, E_atm, IntStor, R_swds, R_mss, R_up

In [95]:
# Taking time step t = 8 as an example.
# The results correspondes with the excel solution.
t1 = PavedRoof(8, 0.508, 0.04987013, 1.596675325)
print(t1.sol())

(1.6, 0.04987013, 1.5501298700000001, 0.504675325, 0.0, 0.0)


Next, use 'while' to loop throughout the whole time steps, and try 2 different sets of coefficients to validate the class function. (Also think about use \__next\__ to iterate)

\textbf{Coefficient set 1}: Default settings: IntStorCap_PavedRoof = 1.6, StormFrac_PavedRoof = 1.0, DiscFrac_PavedRoof = 0.0

In [84]:
t = 1
E_atm = [0]
Int = [0]
IntStor = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

m = PavedRoof(t, P_atm[t], E_pot_OW[t], Init_InStor) # No input parameter means default parameters

while t <= iters-1:
    if t == 1:
        Init_InStor = 0
    # m = PavedRoof(t, P_atm[t], E_pot_OW[t], Init_InStor) # No input parameter means default parameters
    # sol = m.sol()
    sol = m.sol(P_atm[t], E_pot_OW[t])
    Int.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    R_swds.append(sol[3])
    R_mss.append(sol[4])
    R_up.append(sol[5])
    Init_InStor = sol[2]
    # print('time step', t)
    t += 1
filename = 'Class_results_set1.csv'
np.savetxt('sol/' + filename, np.c_[Int, E_atm, IntStor, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Int, E_atm, IntStor, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure
# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
#date_column = pd.DataFrame({'Date': date})
#df = df.merge(date_column, left_index = True, right_index = True)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)
print('The results have been validated.')

The results have been validated.


Coefficient set 2:  Non-default settings: IntStorCap_PavedRoof = 1.3, StormFrac_PavedRoof = 0.5, DiscFrac_PavedRoof = 0.5

### Original code.

In [86]:
t = 1
E_atm = [0]
Int = [0]
IntStor = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

while t <= iters-1:
    if t == 1:
        Init_InStor = 0
    m = PavedRoof(t, P_atm[t], E_pot_OW[t], Init_InStor, IntStorCap_PavedRoof = 1.3, StormFrac_PavedRoof = 0.5, DiscFrac_PavedRoof = 0.5)
    sol = m.sol()
    Int.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    R_swds.append(sol[3])
    R_mss.append(sol[4])
    R_up.append(sol[5])
    Init_InStor = sol[2]
    # print('time step', t)
    t += 1
filename = 'Class_results_set2.csv'
np.savetxt('sol/' + filename, np.c_[Int, E_atm, IntStor, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Int, E_atm, IntStor, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure
# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
#date_column = pd.DataFrame({'Date': date})
#df = df.merge(date_column, left_index = True, right_index = True)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)
print('The results have been validated.')

The results have been validated.


### 3 How to inplement measure into the function.

In [87]:
class PavedRoof:
    def __init__(self, t, P_atm , E_pot_OW, Init_InStor, IntStorCap_PavedRoof = 1.6, StormFrac_PavedRoof = 1.0, DiscFrac_PavedRoof = 0.0):
        self.P_atm = P_atm
        self.init_InStor = Init_InStor 
        self.E_pot_OW = E_pot_OW
        self.IntStorCap = IntStorCap_PavedRoof
        self.StormFrac = StormFrac_PavedRoof
        self.MxdFrac = 1 - StormFrac_PavedRoof
        self.DiscFrac = DiscFrac_PavedRoof

    def __repr__(self):
        return 'Current P is ' + str(self.P_atm) + 'Current E is ' + str(self.E_pot_OW) + '.These are input information.'
        
    def sol(self):
        Int = np.minimum(self.IntStorCap, np.maximum(0, self.P_atm + self.init_InStor))
        E_atm = np.minimum( self.E_pot_OW, Int)
        IntStor = Int - E_atm
        R_swds = self.StormFrac * (1 - self.DiscFrac) * np.maximum(0, self.P_atm - E_atm -( IntStor - self.init_InStor) )
        R_mss = self.MxdFrac * (1 - self.DiscFrac) * np.maximum(0, self.P_atm - E_atm - ( IntStor - self.init_InStor))
        R_up = self.DiscFrac * np.maximum(0, self.P_atm - E_atm - (IntStor - self.init_InStor))
        return Int, E_atm, IntStor, R_swds, R_mss, R_up

In [88]:
t = 1
E_atm = [0]
Int = [0]
IntStor = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

while t <= iters-1:
    if t == 1:
        Init_InStor = 0
    m = PavedRoof(t, P_atm[t], E_pot_OW[t], Init_InStor, IntStorCap_PavedRoof = 1.3, StormFrac_PavedRoof = 0.5, DiscFrac_PavedRoof = 0.5)
    sol = m.sol()
    Int.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    R_swds.append(sol[3])
    R_mss.append(sol[4])
    R_up.append(sol[5])
    Init_InStor = sol[2]
    # print('time step', t)
    t += 1
filename = 'Class_results_set3.csv'
np.savetxt('sol/' + filename, np.c_[Int, E_atm, IntStor, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Int, E_atm, IntStor, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure
# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
#date_column = pd.DataFrame({'Date': date})
#df = df.merge(date_column, left_index = True, righ



t_index = True)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)
print('The results have been validated.')

The results have been validated.
